In [52]:
from src.algo.data import *
from ortools.sat.python import cp_model

In [53]:
model = cp_model.CpModel()
solver = cp_model.CpSolver()

In [77]:
days = ['pon', 'uto', 'sre', 'cet', 'pet']
group_size = 10
classrooms = [0,1,2]

In [78]:
scheduling_input = load_input("../src/algo/input.json")

In [79]:
sessions = generate_sessions(scheduling_input, group_size)
working_hours = [hour for hour in range(scheduling_input.settings.start_hour, scheduling_input.settings.end_hour)]

In [80]:
day_var = {}
slot_var = {}
room_var = {}
flat_time_var = {}
room_time_var = {}
H = len(working_hours)
D = len(days)
R = len(classrooms)
total_slots = H * D

In [81]:
for s in range(len(sessions)):
    day_var[s] = model.NewIntVar(0, D-1, f"day_{s}")
    slot_var[s] = model.NewIntVar(0, H-1, f"hour_{s}")
    room_var[s] = model.NewIntVar(0, R-1, f"classroom{s}")

    flat_time_var[s] = model.NewIntVar(0, total_slots-1, f"flat_time{s}")
    model.Add(flat_time_var[s] == day_var[s] * H + slot_var[s])

    room_time_var[s] = model.NewIntVar(0, total_slots * R - 1, f"room_time_{s}")
    model.Add(room_time_var[s] == room_var[s] * total_slots + flat_time_var[s])
    

In [82]:
model.AddAllDifferent(room_time_var.values())

In [83]:
groups_sessions = {}
for s, session in enumerate(sessions):
    if session.group_id not in groups_sessions:
        groups_sessions[session.group_id] = []
    else:
        groups_sessions[session.group_id].append(s)

for group_id, group_sessions in groups_sessions.items():
    model.AddAllDifferent([room_time_var[s] for s in group_sessions])

In [84]:
solver.Solve(model)

<CpSolverStatus.INFEASIBLE: 3>

In [73]:
solver.Value(slot_var[0])

3